# Physics-Based Physical Validation Layer (Element-Aware)

This notebook implements a clean, element-aware Physical Layer for a cyber-physical power system.

Design Principles:
- Uses normal-operation baseline only
- Continuous z-score severity
- Element-specific aggregation logic
- Breaker physical consistency rule
- No ML involvement

RED ≠ attack.
RED = abnormal physical behaviour.

In [11]:
# =====================================================
# 0. SCENARIO DEFINITIONS
# =====================================================

# -----------------------------------------------------
# A) DATASET GROUND TRUTH (DO NOT MODIFY)
# -----------------------------------------------------

SCENARIO_LOOKUP = {
    # Natural Faults (1–6)
    1: "Fault L1 (10–19%)",
    2: "Fault L1 (20–79%)",
    3: "Fault L1 (80–90%)",
    4: "Fault L2 (10–19%)",
    5: "Fault L2 (20–79%)",
    6: "Fault L2 (80–90%)",

    # Maintenance (13–14)
    13: "Line Maintenance L1",
    14: "Line Maintenance L2",

    # Data Injection Attacks (7–12)
    7:  "Data Injection: L1 Fault 10–19% with tripping",
    8:  "Data Injection: L1 Fault 20–79% with tripping",
    9:  "Data Injection: L1 Fault 80–90% with tripping",
    10: "Data Injection: L2 Fault 10–19% with tripping",
    11: "Data Injection: L2 Fault 20–79% with tripping",
    12: "Data Injection: L2 Fault 80–90% with tripping",

    # Remote Tripping (15–20)
    15: "Remote Tripping: Command Injection R1",
    16: "Remote Tripping: Command Injection R2",
    17: "Remote Tripping: Command Injection R3",
    18: "Remote Tripping: Command Injection R4",
    19: "Remote Tripping: Command Injection R1 & R2",
    20: "Remote Tripping: Command Injection R3 & R4",

    # Relay Setting Change (21–30)
    21: "Relay Setting Change: R1 disabled (L1 10–19% fault)",
    22: "Relay Setting Change: R1 disabled (L1 20–90% fault)",
    23: "Relay Setting Change: R2 disabled (L1 10–49% fault)",
    24: "Relay Setting Change: R2 disabled (L1 50–79% fault)",
    25: "Relay Setting Change: R2 disabled (L1 80–90% fault)",
    26: "Relay Setting Change: R3 disabled (L2 10–19% fault)",
    27: "Relay Setting Change: R3 disabled (L2 20–49% fault)",
    28: "Relay Setting Change: R3 disabled (L2 50–90% fault)",
    29: "Relay Setting Change: R4 disabled (L2 10–79% fault)",
    30: "Relay Setting Change: R4 disabled (L2 80–90% fault)",

    # Relay Setting Change (two relays + fault)
    35: "Relay Setting Change: R1 & R2 disabled (L1 10–49% fault)",
    36: "Relay Setting Change: R1 & R2 disabled (L1 50–90% fault)",
    37: "Relay Setting Change: R3 & R4 disabled (L1 10–49% fault)",
    38: "Relay Setting Change: R3 & R4 disabled (L1 50–90% fault)",

    # Relay Setting Change (maintenance)
    39: "Relay Setting Change: R1 & R2 disabled during maintenance",
    40: "Relay Setting Change: R3 & R4 disabled during maintenance",

    # Normal
    41: "Normal Operation (no disturbances)"
}


# -----------------------------------------------------
# B) INTERPRETATION-LEVEL SCENARIO VIEW
# (Used for system evaluation and discussion)
# -----------------------------------------------------

INTERPRETATION_LOOKUP = SCENARIO_LOOKUP.copy()

# Correct structural interpretation (if needed)
INTERPRETATION_LOOKUP[37] = "RSC: R3 & R4 disabled during L2 fault (10–49%)"
INTERPRETATION_LOOKUP[38] = "RSC: R3 & R4 disabled during L2 fault (50–90%)"
INTERPRETATION_LOOKUP[39] = "RSC: R1 & R2 disabled during maintenance (L1 context)"
INTERPRETATION_LOOKUP[40] = "RSC: R3 & R4 disabled during maintenance (L2 context)"


# -----------------------------------------------------
# C) CATEGORY GROUPING FUNCTION
# -----------------------------------------------------

def scenario_category(marker):
    if marker in range(1, 7):
        return "Natural Fault"
    elif marker in range(7, 13):
        return "Data Injection"
    elif marker in range(15, 21):
        return "Remote Trip"
    elif marker in list(range(21, 31)) + [35, 36, 37, 38, 39, 40]:
        return "Relay Setting Change"
    elif marker in [13, 14]:
        return "Maintenance"
    elif marker == 41:
        return "Normal"
    else:
        return "Other"


In [12]:
import pandas as pd
import numpy as np

## Step 1 — Load Dataset
Ensure your dataset includes a 'marker' column where marker == 41 represents normal operation.

In [13]:
# Update path as needed
DATA_PATH = "../data/merged/multi_class_dataset_clean_FULL.csv"
df = pd.read_csv(DATA_PATH)

NON_RUNTIME = ['marker', 'label', 'label_name', 'marker_name']
FEATURES = [c for c in df.columns if c not in NON_RUNTIME]

print('Total features:', len(FEATURES))

Total features: 132


In [24]:
# --------------------------------------------------
# Select PHYSICAL features only
# --------------------------------------------------

PHYSICAL_TYPES = {
    "voltage_magnitude",
    "current_magnitude",
    "frequency",
    "rocof",
    "impedance",
    "impedance_angle",
    "voltage_phase_angle",
    "current_phase_angle"
}

PHYSICAL_FEATURES = [
    col for col in FEATURES
    if FEATURES[col] in PHYSICAL_TYPES
]

print("Total features:", len(FEATURES))
print("Physical features:", len(PHYSICAL_FEATURES))


TypeError: list indices must be integers or slices, not str

## Step 2 — Normal Baseline Model

In [14]:
normal_mask = df['marker'] == 41

mu = df.loc[normal_mask, FEATURES].mean()
sigma = df.loc[normal_mask, FEATURES].std().replace(0, 1e-6)

print('Baseline computed.')

Baseline computed.


## Step 3 — Continuous Feature Severity

In [15]:
z_scores = (df[FEATURES] - mu) / sigma
severity = (z_scores.abs() / 3).clip(upper=1.0)

severity.head()

,R1-PA1:VH,R1-PM1:V,R1-PA2:VH,R1-PM2:V,R1-PA3:VH,R1-PM3:V,R1-PA4:IH,R1-PM4:I,R1-PA5:IH,R1-PM5:I,...,relay3_log,relay4_log,snort_log1,snort_log2,snort_log3,snort_log4,R1-PA:Z_inf_flag,R2-PA:Z_inf_flag,R3-PA:Z_inf_flag,R4-PA:Z_inf_flag
0,0.234296,1.000000,0.140684,0.965211,0.508643,1.000000,0.225503,0.816916,0.163495,0.924214,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.212612,1.000000,0.161590,1.000000,0.601490,1.000000,0.196283,0.979074,0.187496,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.208374,1.000000,0.165680,1.000000,0.597460,1.000000,0.189305,1.000000,0.192392,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.222855,0.027774,0.151705,0.022066,0.519440,0.023012,0.219439,0.236602,0.165970,0.295353,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.227226,0.122419,0.147543,0.116092,0.515355,0.105866,0.222462,0.250115,0.160781,0.295353,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Step 4 — Define Element Ownership

In [16]:
ELEMENT_FEATURES = {
    'R1': [c for c in FEATURES if c.startswith('R1-')],
    'R2': [c for c in FEATURES if c.startswith('R2-')],
    'R3': [c for c in FEATURES if c.startswith('R3-')],
    'R4': [c for c in FEATURES if c.startswith('R4-')],
    'L1': [c for c in FEATURES if c.startswith(('R1-','R2-')) and ':I' in c],
    'L2': [c for c in FEATURES if c.startswith(('R3-','R4-')) and ':I' in c],
    'G1': [c for c in FEATURES if c.startswith('R1-') and (':F' in c or ':DF' in c)],
    'G2': [c for c in FEATURES if c.startswith('R4-') and (':F' in c or ':DF' in c)],
}

## Step 5 — Element-Specific Aggregation

In [17]:
element_health = pd.DataFrame(index=df.index)

for element, cols in ELEMENT_FEATURES.items():
    score = pd.Series(0.0, index=df.index)

    if element.startswith('G'):  # GENERATOR
        freq_cols = [c for c in cols if ':F' in c]
        rocof_cols = [c for c in cols if ':DF' in c]
        voltage_cols = [c for c in cols if ':V' in c]
        current_cols = [c for c in cols if ':I' in c]

        if freq_cols:
            score = np.maximum(score, severity[freq_cols].max(axis=1))
        if rocof_cols:
            score = np.maximum(score, severity[rocof_cols].max(axis=1))
        if voltage_cols:
            score = np.maximum(score, 0.5 * severity[voltage_cols].max(axis=1))
        if current_cols:
            score = np.maximum(score, 0.5 * severity[current_cols].max(axis=1))

    elif element.startswith('L'):  # LINE
        current_cols = [c for c in cols if ':I' in c]
        voltage_cols = [c for c in cols if ':V' in c]

        if current_cols:
            score = np.maximum(score, severity[current_cols].max(axis=1))
        if voltage_cols:
            score = np.maximum(score, 0.5 * severity[voltage_cols].max(axis=1))

    elif element.startswith('R'):  # RELAY
        current_cols = [c for c in cols if ':I' in c]
        impedance_cols = [c for c in cols if ':Z' in c]

        if current_cols:
            score = np.maximum(score, severity[current_cols].max(axis=1))
        if impedance_cols:
            score = np.maximum(score, severity[impedance_cols].max(axis=1))

    element_health[element] = score

element_health.head()

,R1,R2,R3,R4,L1,L2,G1,G2
0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0
1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0
2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0
3,1.000000,1.000000,1.000000,0.998605,1.000000,1.000000,0.0,0.0
4,0.831566,0.756483,0.752011,0.776701,0.831566,0.776701,0.0,0.0


## Step 6 — Breaker Physical Consistency Rule

In [18]:
for br in ['BR1','BR2','BR3','BR4']:
    status_col = f"{br}:S"
    relay_prefix = br.replace('BR','R')
    current_cols = [c for c in FEATURES if relay_prefix in c and ':I' in c]

    if status_col in df.columns and current_cols:
        current_mag = df[current_cols].abs().max(axis=1)
        is_open = df[status_col] == 0
        breaker_fault = (is_open & (current_mag > 0.05 * current_mag.max()))
        element_health[br] = breaker_fault.astype(float)

element_health.head()

,R1,R2,R3,R4,L1,L2,G1,G2
0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0
1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0
2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0
3,1.000000,1.000000,1.000000,0.998605,1.000000,1.000000,0.0,0.0
4,0.831566,0.756483,0.752011,0.776701,0.831566,0.776701,0.0,0.0


## Step 7 — Baseline Correction

In [19]:
baseline = element_health.loc[normal_mask].median()
element_health = (element_health - baseline).clip(lower=0)

element_health.head()

,R1,R2,R3,R4,L1,L2,G1,G2
0,0.485038,0.499364,0.500156,0.488966,0.467485,0.469205,0.0,0.0
1,0.485038,0.499364,0.500156,0.488966,0.467485,0.469205,0.0,0.0
2,0.485038,0.499364,0.500156,0.488966,0.467485,0.469205,0.0,0.0
3,0.485038,0.499364,0.500156,0.487571,0.467485,0.469205,0.0,0.0
4,0.316604,0.255847,0.252167,0.265667,0.299051,0.245906,0.0,0.0


## Step 8 — Colour Mapping

In [20]:
def colour(score):
    if score < 0.2:
        return 'green'
    elif score < 0.5:
        return 'yellow'
    elif score < 0.8:
        return 'orange'
    else:
        return 'red'

element_colours = element_health.applymap(colour)
element_colours.head()

/var/folders/nv/f9_p0fkx4xsf4ygmqb8ynm9w0000gn/T/ipykernel_18210/2030171503.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  element_colours = element_health.applymap(colour)


,R1,R2,R3,R4,L1,L2,G1,G2
0,yellow,yellow,orange,yellow,yellow,yellow,green,green
1,yellow,yellow,orange,yellow,yellow,yellow,green,green
2,yellow,yellow,orange,yellow,yellow,yellow,green,green
3,yellow,yellow,orange,yellow,yellow,yellow,green,green
4,yellow,yellow,yellow,yellow,yellow,yellow,green,green


In [21]:
# =====================================================
# PHYSICAL LAYER TESTING CELL
# =====================================================

# ----------------------------------
# 1. Attach labels
# ----------------------------------

test_df = element_health.copy()
test_df["marker"] = df["marker"]

# Dataset naming
test_df["dataset_label"] = test_df["marker"].map(SCENARIO_LOOKUP)

# Interpretation naming
test_df["interpretation_label"] = test_df["marker"].map(INTERPRETATION_LOOKUP)

# Category grouping
test_df["category"] = test_df["marker"].apply(scenario_category)

print("Labels attached.\n")


# ----------------------------------
# 2. Compute scenario means
# ----------------------------------

scenario_means = (
    test_df
    .groupby("marker")
    .mean(numeric_only=True)
)

scenario_means["dataset_label"] = scenario_means.index.map(SCENARIO_LOOKUP)
scenario_means["interpretation_label"] = scenario_means.index.map(INTERPRETATION_LOOKUP)
scenario_means["overall_mean"] = scenario_means.drop(
    columns=["dataset_label", "interpretation_label"],
    errors="ignore"
).mean(axis=1)

print("Scenario-level means computed.\n")


# ----------------------------------
# 3. Category-level summary
# ----------------------------------

category_means = (
    test_df
    .groupby("category")
    .mean(numeric_only=True)
)

category_means["overall_mean"] = category_means.mean(axis=1)

print("Category-level summary:\n")
print(category_means["overall_mean"].sort_values())
print("\n")


# ----------------------------------
# 4. Automatic sanity checks
# ----------------------------------

print("===== SANITY CHECKS =====")

normal_score = category_means.loc["Normal", "overall_mean"]
print(f"Normal mean: {normal_score:.4f}")

if "Remote Trip" in category_means.index:
    remote_score = category_means.loc["Remote Trip", "overall_mean"]
    print(f"Remote Trip mean: {remote_score:.4f}")
else:
    remote_score = None

if "Natural Fault" in category_means.index:
    fault_score = category_means.loc["Natural Fault", "overall_mean"]
    print(f"Natural Fault mean: {fault_score:.4f}")
else:
    fault_score = None

print("\nExpected behaviour:")
print("- Normal should be near 0")
print("- Remote Trip should be low")
print("- Natural Fault should be highest")

# Basic logic validation
if fault_score and remote_score:
    if fault_score > remote_score:
        print("\n✓ Physical layer separates faults from cyber-only events.")
    else:
        print("\n⚠ Warning: Physical layer may not be separating correctly.")

print("\nTesting complete.")


Labels attached.

Scenario-level means computed.

Category-level summary:

category
Natural Fault           0.617359
Data Injection          1.260824
Maintenance             1.688712
Remote Trip             2.144866
Relay Setting Change    3.583639
Normal                  4.601663
Name: overall_mean, dtype: float64


===== SANITY CHECKS =====
Normal mean: 4.6017
Remote Trip mean: 2.1449
Natural Fault mean: 0.6174

Expected behaviour:
- Normal should be near 0
- Remote Trip should be low
- Natural Fault should be highest

⚠ Warning: Physical layer may not be separating correctly.

Testing complete.
